In [31]:
from dotenv import load_dotenv
load_dotenv()

True

In [32]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langchain.agents import create_agent

In [33]:
loader = PyPDFLoader("../data/sample.pdf")
docs =  loader.load()
len(docs)

23

Text Splitting

In [34]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

54

Embeddings and VectorDb store

In [35]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3424.14it/s]


In [36]:
@tool
def retrieve_relevant_docs(query:str):
    """
    This tool retrieves relevant douments from the vector store based on the query provided by the user. It uses the vector store's similarity search to find documents that are semantically similar to the query.
    
    """
    docs = vector_store.similarity_search(query=query, k=4)
    context = ""
    for doc in docs:
        context += doc.page_content + "\n\n"
    return context


In [37]:
retriever_tool = retrieve_relevant_docs
retriever_tool.invoke({"query": "what is the main topic of the document?"})

'Page 9 of 23 \n1.5 Reference Models \nA reference model is a conceptual framework that describes the functions a networking system should \nperform, without specifying exactly how. The two most important reference models are the OSI model \nand the TCP/IP model. \n(a) The OSI Reference Model \nOSI stands for Open Systems Interconnection. It was developed by ISO (International Standards \nOrganization) as a first step towards international standardisation. It has 7 layers, each with a well -\ndefined function. \nLayer \nNo. \nLayer Name Main Function \n7 Application Provides network services directly to end -user \napplications (e-mail, file transfer, browsing) \n6 Presentation Data translation, encryption/decryption, and \ncompression (formatting so both ends understand) \n5 Session Establishes, manages, and terminates sessions (dialogue \ncontrol) between two communicating hosts \n4 Transport End-to-end delivery of complete messages; error \ncontrol and flow control between source an

In [38]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [39]:
System_prompt = """You are a helpful assistant that answers questions based on the context provided. If the answer is not contained within the context, politely respond that you don't know. Always use the 'retriever_tool' for questions requiring external knowledge. Do not make up answers. If the context is empty, respond that you don't know."""

In [40]:
agent = create_agent(
    model=llm,
    tools=[retrieve_relevant_docs],
    system_prompt=System_prompt
)

In [44]:
query = "What are reference models?"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})

In [45]:
result = response["messages"][-1].content

In [46]:
print(result)

**Reference models** are conceptual frameworks that describe *what* functions a networking system should perform, but not *how* those functions are implemented. They provide a high‑level, standardized way of thinking about network architecture, helping designers, engineers, and educators talk about and analyze systems in a consistent manner.

The two most widely taught and used reference models are:

| Model | Origin | Layers | Purpose |
|-------|--------|--------|---------|
| **OSI (Open Systems Interconnection)** | ISO (International Organization for Standardization) | 7 layers: <br>1. Physical<br>2. Data Link<br>3. Network<br>4. Transport<br>5. Session<br>6. Presentation<br>7. Application | Provides a theoretical, modular framework for understanding how data moves from one system to another. |
| **TCP/IP** | Developed around the ARPANET, later adopted by the Internet | 4 layers: <br>1. Link (Network Interface)<br>2. Internet<br>3. Transport<br>4. Application | Serves as the practica